# Step 4 · Where to build? A greedy heuristic (MLCC) vs a genetic algorithm (Grenoble 2026)

**Question:** with a given budget, which streets of Grenoble should be turned into comfortable (LTS 1) cycling
streets to maximise the predicted bike share?

* **Decision:** each street with LTS 2–5 is either left as it is or upgraded **directly to LTS 1**, at a cost per
  metre that depends on its current LTS (175 / 225 / 240 / 250 €/m for LTS 2 / 3 / 4 / 5). LTS 5 roads where cycling
  is prohibited cannot be upgraded.
* **Budget:** 25 % of the cost of upgrading the whole network.
* **Score of a plan:** the predicted share of the step-3 model (Grenoble, 22.82 % today).

Two algorithms are compared:

1. **MLCC** — *Maximise the Largest Connected Component*: first connect the LTS 1 islands with the shortest
   paths, then grow the LTS 1 network outward, stopping at the first street that does not fit in the budget.
   Fast and transparent.
2. **Genetic algorithm (GA)** — a population of 30 plans evolves for 30 generations (selection, crossover,
   mutation). The MLCC plan is included in the first generation, so the GA can only match or beat it.

| | |
|---|---|
| **Input** | `results/lts_networks/grenoble/<run>/2026/grenoble_lts_2026_SIMPLE.pkl` (step 1) |
| **Output** | `results/ga_vs_mlcc/` — comparison table, GA history, selected edges, upgraded networks (.pkl, not in git) |
| **Run time** | ~1 min for MLCC, ~10–15 min for the GA (set `POPULATION_SIZE=10, N_GENERATIONS=5` for a quick test) |

Code: [`cyclelts/optimization.py`](../cyclelts/optimization.py). No optimality is claimed for either method.

In [1]:
# Setup: find the repository root and make the `cyclelts` package importable.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cyclelts").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from cyclelts import config
from cyclelts.mode_share import GRENOBLE_MODEL, GRENOBLE_2026_SHARE
from cyclelts.optimization import UpgradeProblem, COSTS_GA_COMPARISON
from cyclelts.metrics import edge_level as level
from cyclelts.plotting import plot_solutions

BUDGET_FRACTION = 0.25           # share of the cost of upgrading the whole network
POPULATION_SIZE, N_GENERATIONS = 30, 30
GA_SETTINGS = dict(crossover_rate=0.85, mutation_rate=0.02, elite_size=2, tournament_size=3, random_seed=42)
OUTPUT = config.GA_VS_MLCC_DIR
OUTPUT.mkdir(parents=True, exist_ok=True)

G = config.grenoble_graph(2026)
problem = UpgradeProblem(G, COSTS_GA_COMPARISON, GRENOBLE_MODEL)
BUDGET_EUR = BUDGET_FRACTION * problem.full_cost_all
print(f"Network: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges | upgradeable edges: {len(problem.gene_edges):,}")
print(f"Cost of upgrading everything (incl. prohibited roads): €{problem.full_cost_all:,.0f}")
print(f"Cost of upgrading every eligible edge:                 €{problem.full_cost_eligible:,.0f}")
print(f"Budget ({BUDGET_FRACTION:.0%}):                                       €{BUDGET_EUR:,.0f}")

Network: 5,664 nodes, 7,729 edges | upgradeable edges: 3,843
Cost of upgrading everything (incl. prohibited roads): €49,751,557
Cost of upgrading every eligible edge:                 €42,296,247
Budget (25%):                                       €12,437,889


## 1. Baseline: today's network

In [3]:
BASELINE = GRENOBLE_MODEL.metrics_and_share(G)
display(pd.DataFrame([BASELINE], index=["Baseline"]).round(5))
assert abs(BASELINE["predicted_share_percent"] - GRENOBLE_2026_SHARE) < 0.005, "Baseline should be 22.820 %"

,cycling_length_km,normalized_network_length,weighted_lcc_normalized,lcc_lts1_km,lcc_lts2_km,lcc_lts3_km,lcc_lts4_km,indicator_I,predicted_share_percent
Baseline,418.26236,1.0,0.40742,47.05748,381.91738,415.75026,418.26236,0.64445,22.82022


## 2. MLCC heuristic

In [4]:
start = time.perf_counter()
MLCC_CHROMOSOME, G_mlcc, MLCC_STATE = problem.run_mlcc(G, BUDGET_EUR, verbose=True)
MLCC_METRICS = GRENOBLE_MODEL.metrics_and_share(G_mlcc)
print(f"MLCC done in {time.perf_counter()-start:.1f}s — {MLCC_STATE['stop_reason']}")
print(f"Spent €{problem.chromosome_cost(MLCC_CHROMOSOME):,.0f} of €{BUDGET_EUR:,.0f}; upgraded edges: {int(MLCC_CHROMOSOME.sum()):,}")
display(pd.DataFrame([BASELINE, MLCC_METRICS], index=["Baseline", "MLCC"]).round(5))

MLCC phase LTS1 connection 0, spent €0


MLCC phase LTS1 connection 50, spent €2,532,716


MLCC phase LTS1 connection 100, spent €3,503,380


MLCC phase LTS1 connection 150, spent €4,370,821


MLCC phase LTS1 connection 200, spent €5,292,461


MLCC phase LTS1 connection 250, spent €5,992,804


... (4 outputs left out)

MLCC phase LTS1 connection 500, spent €9,076,311


MLCC phase LTS1 connection 550, spent €9,538,347


MLCC done in 45.0s — Insufficient budget for edge 859106-859110
Spent €12,421,562 of €12,437,889; upgraded edges: 1,353


,cycling_length_km,normalized_network_length,weighted_lcc_normalized,lcc_lts1_km,lcc_lts2_km,lcc_lts3_km,lcc_lts4_km,indicator_I,predicted_share_percent
Baseline,418.26236,1.0,0.40742,47.05748,381.91738,415.75026,418.26236,0.64445,22.82022
MLCC,418.26236,1.0,0.74024,258.23383,394.95486,417.31245,418.26236,0.84414,34.78188


## 3. Genetic algorithm (seeded with the MLCC plan)

In [5]:
start = time.perf_counter()
GA_CHROMOSOME, GA_HISTORY = problem.run_ga(BUDGET_EUR, MLCC_CHROMOSOME, population_size=POPULATION_SIZE,
                                           n_generations=N_GENERATIONS, **GA_SETTINGS)
G_ga = problem.to_graph(GA_CHROMOSOME)
GA_METRICS = GRENOBLE_MODEL.metrics_and_share(G_ga)
assert problem.chromosome_cost(GA_CHROMOSOME) <= BUDGET_EUR + 1e-7
assert GA_METRICS["predicted_share_percent"] + 1e-9 >= MLCC_METRICS["predicted_share_percent"]
print(f"GA done in {time.perf_counter()-start:.1f}s | distinct plans evaluated: {problem.ga_fitness_evaluations:,}")

GA   0/30: best 34.78188% | mean 25.70147% | cost €12,421,562


GA   1/30: best 34.78188% | mean 26.86225% | cost €12,421,562


GA   2/30: best 34.78188% | mean 28.39581% | cost €12,421,562


GA   3/30: best 34.78188% | mean 30.22472% | cost €12,421,562


GA   4/30: best 34.78188% | mean 30.88125% | cost €12,421,562


GA   5/30: best 34.78188% | mean 31.17102% | cost €12,421,562


... (22 outputs left out)

GA  28/30: best 34.78188% | mean 31.96011% | cost €12,421,562


GA  29/30: best 34.78188% | mean 32.03518% | cost €12,421,562


GA  30/30: best 34.78188% | mean 32.09183% | cost €12,421,562


GA done in 682.4s | distinct plans evaluated: 840


## 4. Comparison

With this budget the GA does not find a better plan than MLCC: the greedy "connect and grow" strategy is
already very good for this objective, at a tiny fraction of the computing time.

In [6]:
rows = []
for name, metric, chosen in [("Baseline", BASELINE, np.zeros(len(problem.gene_edges), dtype=bool)),
                             ("MLCC", MLCC_METRICS, MLCC_CHROMOSOME), ("Genetic algorithm", GA_METRICS, GA_CHROMOSOME)]:
    rows.append({"solution": name, "upgrade_edges": int(chosen.sum()), "spent_eur": problem.chromosome_cost(chosen),
                 "budget_eur": BUDGET_EUR,
                 "mandatory_lts5_upgraded": sum(bool(chosen[i]) and level(G[u][v]) == 5 and bool(G[u][v].get("mandatory_upgrade_to_lts1", False))
                                                for i, (u, v) in enumerate(problem.gene_edges)),
                 **metric})
comparison = pd.DataFrame(rows)
display(comparison[["solution", "upgrade_edges", "spent_eur", "lcc_lts1_km", "weighted_lcc_normalized",
                    "predicted_share_percent"]].round(3))

fig = go.Figure()
fig.add_scatter(x=GA_HISTORY.generation, y=GA_HISTORY.best_share_percent, mode="lines+markers", name="GA best")
fig.add_scatter(x=GA_HISTORY.generation, y=GA_HISTORY.mean_share_percent, mode="lines", name="GA population mean")
fig.add_hline(y=MLCC_METRICS["predicted_share_percent"], line_dash="dash", annotation_text="MLCC")
fig.add_hline(y=BASELINE["predicted_share_percent"], line_dash="dot", annotation_text="Baseline")
fig.update_layout(template="plotly_white", height=450, title="Predicted Grenoble bike share — GA convergence",
                  xaxis_title="Generation", yaxis_title="Predicted share (%)")
fig.show()

,solution,upgrade_edges,spent_eur,lcc_lts1_km,weighted_lcc_normalized,predicted_share_percent
0,Baseline,0,0.000000e+00,47.057,0.407,22.820
1,MLCC,1353,1.242156e+07,258.234,0.740,34.782
2,Genetic algorithm,1353,1.242156e+07,258.234,0.740,34.782


## 5. Maps (upgraded streets in blue)

In [7]:
plot_solutions([G, problem.to_graph(MLCC_CHROMOSOME, "mlcc"), G_ga], ["Baseline", "MLCC", "GA"]);

## 6. Save

In [8]:
comparison.to_csv(OUTPUT / "comparison.csv", index=False)
GA_HISTORY.to_csv(OUTPUT / "ga_history.csv", index=False)
pd.DataFrame(MLCC_STATE["log"]).to_csv(OUTPUT / "mlcc_upgrades.csv", index=False)
for name, mask in [("mlcc", MLCC_CHROMOSOME), ("ga", GA_CHROMOSOME)]:
    problem.selected_edges_table(mask).to_csv(OUTPUT / f"{name}_selected_edges.csv", index=False)
    config.save_pickle(problem.to_graph(mask, algorithm=name), OUTPUT / f"grenoble_2026_{name}_optimized.pkl")
print("Saved in", OUTPUT.relative_to(ROOT))

Saved in results\ga_vs_mlcc
